# Hidden Markov Model Pipeline

End-to-end notebook for preprocessing-aware HMM training, evaluation, and inference.

In [1]:
!rm -rf ASP-GRP4-ANIMAL-SOUND-DETECTION
!git clone -b hidden-markov-model https://github.com/HamzaElhmd/ASP-GRP4-ANIMAL-SOUND-DETECTION.git
!cd ASP-GRP4-ANIMAL-SOUND-DETECTION/ && pip install -r requirements.txt
!!pip install --force-reinstall torchcodec

Cloning into 'ASP-GRP4-ANIMAL-SOUND-DETECTION'...
remote: Enumerating objects: 8813, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 8813 (delta 89), reused 213 (delta 49), pack-reused 8556 (from 3)
Receiving objects: 100% (8813/8813), 575.22 MiB | 19.31 MiB/s, done.
Resolving deltas: 100% (1118/1118), done.
Updating files: 100% (8813/8813), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

['Collecting torchcodec',
 '  Downloading torchcodec-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (8.2 kB)',
 'Downloading torchcodec-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl (3.0 MB)',
 '\x1b[?25l   \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.0/3.0 MB\x1b \x1b?\x1b eta \x1b-:--:--\x1b',
 '\x1b[2K   \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b3.0/3.0 MB\x1b \x1b103.9 MB/s\x1b eta \x1b0:00:00\x1b',
 '\x1b[?25hInstalling collected packages: torchcodec',
 '  Attempting uninstall: torchcodec',
 '    Found existing installation: torchcodec 0.0.0.dev0',
 '    Uninstalling torchcodec-0.0.0.dev0:',
 '      Successfully uninstalled torchcodec-0.0.0.dev0',
 'Successfully installed torchcodec-0.14.0']

In [1]:
from pathlib import Path
import os
import sys

import torch

PROJECT_ROOT = Path("/content/ASP-GRP4-ANIMAL-SOUND-DETECTION")
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(PROJECT_ROOT)
print(os.getcwd())
print(sys.path[0])
print(DEVICE)

/content/ASP-GRP4-ANIMAL-SOUND-DETECTION
/content/ASP-GRP4-ANIMAL-SOUND-DETECTION
/content/ASP-GRP4-ANIMAL-SOUND-DETECTION
cuda


In [2]:
from src.audit import create_dataset_metadata, dataset_statistics, FULL_DATASET
from src.preprocessor import preprocess_dataset
from src.hmm import train_hmm_suite, evaluate_suite, infer_continuous_file, BinaryHMMConfig

print('Imports ready')

Imports ready


## 1. Audit and preprocessing

Run this if `dataset/farmyard.csv` or `processed/farmyard.csv` are missing or stale.

In [3]:
raw_manifest = create_dataset_metadata()
display(raw_manifest.head())
display(dataset_statistics(raw_manifest))

,filepath,n_samples,channel,samplerate,duration,label
0,dataset/sheep/koyun_20.wav,29952,mono,24000,1.248000,sheep
1,dataset/sheep/4-188703-C-8.wav,220500,mono,44100,5.000000,sheep
2,dataset/sheep/koyun_8.wav,36186,mono,11025,3.282177,sheep
3,dataset/sheep/koyun_35.wav,47188,mono,11025,4.280091,sheep
4,dataset/sheep/koyun_15.wav,13375,mono,11025,1.213152,sheep


,num_samples,total_duration,mean_duration,std_duration,min_duration,max_duration,mono_samples,stereo_samples,unique_samplerates
label,,,,,,,,,
background,150,673.270000,4.488467,0.620812,1.380000,5.000000,81,69,3
cat,240,1593.539471,6.639748,4.596365,0.498413,17.976000,236,4,5
cow,115,393.392697,3.420806,1.749038,0.767619,8.559875,112,3,7
dog,240,1020.796023,4.253317,3.207035,0.241270,17.200187,223,17,10
rooster,80,400.533333,5.006667,0.007969,5.000000,5.022766,80,0,1
sheep,83,362.078951,4.362397,6.522211,0.454966,60.000000,77,6,8


In [4]:
processed_manifest = preprocess_dataset()
display(processed_manifest.head())

Before augmentation balance:
label
background    150
cat           240
cow           115
dog           240
rooster        80
sheep          83
After preprocessing balance:
label
background    1137
cat           2652
cow            756
dog           1896
rooster        720
sheep          666


,filepath,n_samples,channel,samplerate,duration,label,source_file,segment_index,segment_start_sample,segment_end_sample,segment_type,augmentation_index,augmentation_tags,augmentation_group
0,processed/sheep/koyun_20_frame_000.wav,32000,mono,16000,2.0,sheep,koyun_20.wav,0,0,19968,training_frame,NaN,NaN,NaN
1,processed/sheep/koyun_20_aug_000_frame_000.wav,32000,mono,16000,2.0,sheep,koyun_20.wav,0,0,19968,augmentation_frame,0.0,bgmix_484|shift_3791|gain_5.35db,koyun_20_aug_000
2,processed/sheep/koyun_20_aug_001_frame_000.wav,32000,mono,16000,2.0,sheep,koyun_20.wav,0,0,19968,augmentation_frame,1.0,bgmix_544|shift_-1043|gain_-5.35db,koyun_20_aug_001
3,processed/sheep/4-188703-C-8_frame_000.wav,32000,mono,16000,2.0,sheep,4-188703-C-8.wav,0,0,32000,training_frame,NaN,NaN,NaN
4,processed/sheep/4-188703-C-8_frame_001.wav,32000,mono,16000,2.0,sheep,4-188703-C-8.wav,1,32000,64000,training_frame,NaN,NaN,NaN


## 2a. Configure and Train the HMM Suite

This cell configures the HMM training process. You can experiment with the hyperparameters in `BinaryHMMConfig` to improve model performance.

- `device`: Set to `'cuda'` to use the GPU for training. This will significantly speed up the process.
- `n_iter`: The number of training iterations for the HMM. More iterations can lead to a better fit, but will take longer.
- `n_components`: The number of Gaussian mixture components for each HMM state. A higher number can model more complex sounds but may overfit.
- `max_train_sequences_per_class`:  Limits the number of training samples per class. This is useful for quick experiments on a smaller subset of the data.

In [5]:
# --- Configure the training ---
config = BinaryHMMConfig(
    n_iter=10,
    n_components=3,
    device='cuda',
    verbose=True,
    max_train_sequences_per_class=1000,
)

print("Training with config:", config)

# --- Run the training ---
train_metrics = train_hmm_suite(
    processed_root=Path('processed'),
    output_dir=Path('artifacts/hmm'),
    config=config
)

train_metrics

[train] split sizes train=5502 val=1182 test=1143
[train] feature device=cpu
[train] extracting features for dog
[train] extracted dog in 218.7s
[train] extracting features for cat
[train] extracted cat in 306.5s
[train] extracting features for sheep
[train] extracted sheep in 76.5s
[train] extracting features for cow
[train] extracted cow in 86.8s
[train] extracting features for rooster
[train] extracted rooster in 82.9s
[train] extracting features for background
[train] extracted background in 131.5s
[train] fitting scaler
[train] fitting class models
[train:dog] scaling sequences
[train:cat] scaling sequences
[train:sheep] scaling sequences
[train:cow] scaling sequences
[train:dog] capped train seqs to 500
[train:cow] capped train seqs to 500
[train:rooster] scaling sequences
[train:background] scaling sequences
[train:cat] capped train seqs to 500
[train:background] capped train seqs to 500
[train:rooster] capped train seqs to 500
[train:rooster] fitting model on 500 sequences
[hmm

{'splits': {'train': 5502, 'val': 1182, 'test': 1143},
 'classes': {'dog': {'validation_log_likelihood_mean': -10006.103499538214,
   'test_log_likelihood_mean': -9803.342290900735},
  'cat': {'validation_log_likelihood_mean': -10151.287853582975,
   'test_log_likelihood_mean': -9884.01427883572},
  'sheep': {'validation_log_likelihood_mean': -9159.928962953629,
   'test_log_likelihood_mean': -9172.194438416282},
  'cow': {'validation_log_likelihood_mean': -7402.504544194539,
   'test_log_likelihood_mean': -7808.829833676116},
  'rooster': {'validation_log_likelihood_mean': -9298.613807395652,
   'test_log_likelihood_mean': -8205.44325934516},
  'background': {'validation_log_likelihood_mean': -9011.167454310826,
   'test_log_likelihood_mean': -9265.39595249721}}}

## 3. Evaluate on the held-out split

This cell evaluates the trained models on the held-out test set. It calculates frame-based and event-based metrics.

The `collar_seconds` parameter in `evaluate_suite` defines the tolerance for matching predicted events to ground truth events.

In [7]:
# Clear the cache to ensure the correct device is used for evaluation
hmm._MFCC_TRANSFORM_CACHE.clear()

diagnostics = hmm.evaluate_suite(
    model_dir=Path("artifacts/hmm"),
    processed_root=Path("processed"),
    output_json=Path("artifacts/hmm/final_diagnostics.json"),
    collar_seconds=0.5
)
diagnostics

{'frame_based': {'dog': {'precision': 1.0,
   'recall': 0.5807504727779437,
   'f1': 0.734778173758642},
  'cat': {'precision': 1.0,
   'recall': 0.4850770821583004,
   'f1': 0.6532685582263859},
  'sheep': {'precision': 1.0,
   'recall': 0.36335150717553427,
   'f1': 0.5330268903700299},
  'cow': {'precision': 1.0,
   'recall': 0.7373737373737373,
   'f1': 0.8488372093023255},
  'rooster': {'precision': 1.0,
   'recall': 0.6524722692235383,
   'f1': 0.7896922464303999},
  'background': {'precision': 1.0,
   'recall': 0.30701595358955763,
   'f1': 0.4697967958943061}},
 'event_based': {'dog': {'precision': 0.058823529411764705,
   'recall': 0.17254901960784313,
   'f1': 0.08773678963110668},
  'cat': {'precision': 0.03557910673732021,
   'recall': 0.1087962962962963,
   'f1': 0.05362236166571591},
  'sheep': {'precision': 0.0035211267605633804,
   'recall': 0.012345679012345678,
   'f1': 0.005479452054794521},
  'cow': {'precision': 0.2905405405405405,
   'recall': 0.43434343434343436,

## 4a. Configure and Run Inference

This cell runs the inference on an unseen audio file. You can tune the post-processing parameters to improve the event detection accuracy.

- `threshold`: The probability threshold for a frame to be considered 'active'.
- `median_width`: The width of the median filter used to smooth the predictions.
- `gap_fill_ms`: The maximum duration in milliseconds to fill a gap between two events of the same class.

In [8]:
# --- Configure the inference ---
UNSEEN_WAV = Path('test_scene.wav')

# --- Experiment with these parameters ---
events = infer_continuous_file(
    UNSEEN_WAV,
    model_dir=Path('artifacts/hmm'),
    threshold=0.6,          # Detection threshold
    median_width=7,         # Smoothing filter width
    gap_fill_ms=400,        # Stitching gap for events
    verbose=True            # Set to True to see the inference performance summary
)

events

[{'event_start': '0.000', 'event_end': '59.970', 'animal': 'background'},
 {'event_start': '0.000', 'event_end': '5.580', 'animal': 'cat'},
 {'event_start': '0.000', 'event_end': '59.970', 'animal': 'cow'},
 {'event_start': '0.000', 'event_end': '59.970', 'animal': 'dog'},
 {'event_start': '0.000', 'event_end': '59.970', 'animal': 'rooster'},
 {'event_start': '0.000', 'event_end': '59.970', 'animal': 'sheep'},
 {'event_start': '6.940', 'event_end': '7.220', 'animal': 'cat'},
 {'event_start': '7.540', 'event_end': '7.570', 'animal': 'cat'},
 {'event_start': '7.940', 'event_end': '10.800', 'animal': 'cat'},
 {'event_start': '11.150', 'event_end': '12.220', 'animal': 'cat'},
 {'event_start': '13.150', 'event_end': '13.970', 'animal': 'cat'},
 {'event_start': '14.850', 'event_end': '16.470', 'animal': 'cat'},
 {'event_start': '17.000', 'event_end': '17.750', 'animal': 'cat'},
 {'event_start': '18.190', 'event_end': '18.520', 'animal': 'cat'},
 {'event_start': '19.190', 'event_end': '19.440

In [9]:
from src.hmm import save_inference_json

save_inference_json(events, Path('result_hmm.json'))
print('Wrote result_hmm.json')

Wrote result_hmm.json
